# AMS-02 Proton & Helium Walkthrough

End-to-end demonstration: CSV ingestion, schema harmonisation,
diagonal likelihood construction, and parquet export.

## 1. Imports

Load modules from the ams02wb package: parsers, schema, harmoniser, likelihood, and exports.

In [ ]:
import io
import json
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd

from ams02wb.parsers.csv_parser import parse_csv
from ams02wb.schema.models import Measurement, UncertaintyLabel, ProvenanceRecord
from ams02wb.harmoniser.pipeline import run_harmonisation_pipeline
from ams02wb.likelihood.diagonal import build_diagonal_covariance, UNCERTAINTY_LABEL, MODE
from ams02wb.exports.parquet import export_parquet

## 2. Load proton sample CSV

Read the sample CSV using the generic `parse_csv` interface and inspect the first few rows.

In [ ]:
proton_path = Path("../data/samples/proton_sample.csv")
with open(proton_path, encoding="utf-8") as fh:
    proton_records = parse_csv(fh)

print(f"Parsed {len(proton_records)} proton rows")
proton_records[:3]

## 3. Inspect provenance_json

Each row carries a `provenance_json` field that traces the data back to its source paper and table.

In [ ]:
for row in proton_records[:3]:
    prov = row["provenance_json"]
    print(f"provenance_json: {prov}")

## 4. Build Measurement objects from parsed records

Convert flat dicts to `Measurement` instances with asymmetric errors for the harmoniser.

In [ ]:
from ams02wb.parsers.context import ParseContext

def records_to_measurements(records, species="PROTON"):
    """Convert parse_csv output to Measurement list."""
    measurements = []
    for r in records:
        m = Measurement(
            energy_low=r["x_min"],
            energy_high=r["x_max"],
            energy_mid=(r["x_min"] + r["x_max"]) / 2.0,
            value=r["y_value"],
            species=species,
            stat_err_pos=r["stat_err"],
            stat_err_neg=r["stat_err"],
            sys_err_pos=r["sys_err_total"],
            sys_err_neg=r["sys_err_total"],
        )
        measurements.append(m)
    return measurements

proton_measurements = records_to_measurements(proton_records, species="PROTON")
print(f"{len(proton_measurements)} Measurement objects created")
proton_measurements[0]

## 5. Run harmonisation pipeline

Apply species normalisation, axis harmonisation, uncertainty labelling,
and time-window normalisation in fixed order.

In [ ]:
parse_ctx = ParseContext(stat_err_from_table=True, sys_err_from_table=True)

provenance = {
    "paper_doi": "10.1103/PhysRevLett.114.171103",
    "table_id": "T1",
    "file_url": "data/samples/proton_sample.csv",
    "source_type": "csv",
}

proton_canonical = run_harmonisation_pipeline(
    proton_measurements, parse_ctx, provenance
)
print(f"{len(proton_canonical)} canonical records")
proton_canonical[0]

## 6. Build diagonal covariance matrix

Use published stat and sys errors to construct a diagonal covariance.
The `uncertainty_label` for this mode is 'published'.

In [ ]:
stat_arr = np.array([r["stat_err"] for r in proton_canonical])
sys_arr = np.array([r["sys_err_total"] for r in proton_canonical])

cov = build_diagonal_covariance(stat_arr, sys_arr)
print(f"Covariance shape: {cov.shape}")
print(f"uncertainty_label: {UNCERTAINTY_LABEL}")
print(f"mode: {MODE}")
print(f"First 3 diagonal entries: {np.diag(cov)[:3]}")

## 7. Assemble fit-ready dataset and export to parquet

In [ ]:
from ams02wb.likelihood.fitready import build_fit_dataset

y = np.array([r["y_value"] for r in proton_canonical])
x = np.array([r["x_centre"] for r in proton_canonical])

fit_ds = build_fit_dataset(
    y=y, x=x, covariance=cov,
    uncertainty_label=UNCERTAINTY_LABEL,
    mode=MODE,
    provenance=provenance,
    species="PROTON",
    x_axis_type="kinetic_energy_per_nucleon",
    y_unit="m-2 sr-1 s-1 GV-1",
)
print(f"Fit dataset keys: {sorted(fit_ds.keys())}")
print(f"n_points: {fit_ds['n_points']}")

## 8. Export to parquet and read back

Write the proton dataset to parquet, then read it back and verify the row count matches.

In [ ]:
export_df = pd.DataFrame({
    "x_centre": fit_ds["x"],
    "y_value": fit_ds["y"],
    "stat_err": stat_arr,
    "sys_err_total": sys_arr,
})

export_dataset = {
    "data": export_df,
    "provenance": provenance,
    "covariance_label": UNCERTAINTY_LABEL,
}

tmpdir = tempfile.mkdtemp()
parquet_path = Path(tmpdir) / "proton_fit.parquet"
written = export_parquet(export_dataset, parquet_path)
print(f"Exported to: {written}")

In [ ]:
readback = pd.read_parquet(written)
print(f"Read back {len(readback)} rows from parquet")
assert len(readback) == len(proton_records), (
    f"Row count mismatch: parquet has {len(readback)}, input had {len(proton_records)}"
)
print("Row count assertion passed.")
readback.head()